# Stage 2 – Select exactly 6 customers (Quantum / Hybrid Binary Optimisation)

**Problem:** from the top-10 customers (by predicted probability, ties → `customer_id` ascending) choose **exactly 6** to maximise total expected net value, subject to  
`Σ cost·x ≤ 25` (budget) and `Σ capacity·x ≤ 22` (capacity).

**Pipeline (fully automatic – just *Run All*):**

| # | Solver | Type |
|---|--------|------|
| 0 | Exact brute force (C(10,6)=210 subsets) | classical reference |
| 1 | QUBO with slack bits + Simulated Annealing | annealing-style QUBO solver |
| 2 | QAOA (Dicke start + XY mixer) | variational |
| 3 | CVaR-QAOA | variational |
| 4 | Warm-Start QAOA (initial state seeded from the annealer) | variational / hybrid |
| 5 | Grover Adaptive Search (amplitude amplification) | search |
| 6 | Quantum-inspired evolutionary algorithm (QIEA) | classical-driven |
| 7 | XY-QAOA: finite-shot, noisy, and noisy + readout-mitigated | NISQ realism |
| 8 | Standard transverse-field X-mixer penalised-QUBO QAOA (1024-state space) | comparison baseline |

Every candidate is **validated** (6 unique IDs, all in top-10, both limits). The best valid one is the final answer.

## 1. Configuration & data loading

In [1]:
import os, itertools, math, time, warnings
import numpy as np, pandas as pd
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

def find(name):
    for d in [".", "/mnt/user-data/uploads", "/content", "/kaggle/input", ".."]:
        p = os.path.join(d, name)
        if os.path.exists(p): return p
    raise FileNotFoundError(name)

sub  = pd.read_csv("/kaggle/input/datasets/mayankkumar2177/quant-files/submission.csv")
data = pd.read_excel("/kaggle/input/datasets/mayankkumar2177/quant-files/stage_2_data.xlsx")
cons = pd.read_excel("/kaggle/input/datasets/mayankkumar2177/quant-files/stage_2_cons.xlsx")

# --- constraints read from the file (fallback to the known values) ---
cmap = dict(zip(cons["symbol"].astype(str), cons["value"]))
N = int(cmap.get("N", 10)); K = int(cmap.get("K", 6))
BUDGET = float(cmap.get("C", 25)); CAPACITY = float(cmap.get("H", 22))
print(f"N={N}, K={K}, budget<={BUDGET}, capacity<={CAPACITY}")

# --- top-N candidates (ties: customer_id ascending) ---
top = (sub.sort_values(["probability", "customer_id"], ascending=[False, True])
          .head(N).merge(data, on="customer_id", how="left").reset_index(drop=True))
assert top["customer_id"].is_unique and top.isna().sum().sum() == 0

# expected net value = P(deposit) * expected_deposit_value - contact_cost
# (assumption: the brief does not spell the formula out; change here if your rubric differs)
top["net_value"] = top["probability"] * top["expected_deposit_value"] - top["contact_cost"]
top["bit_index"] = range(N)
ids  = top["customer_id"].tolist()
v    = top["net_value"].to_numpy(float)
cost = top["contact_cost"].to_numpy(float)
cap  = top["capacity_units"].to_numpy(float)
top[["bit_index","customer_id","probability","contact_cost","expected_deposit_value",
     "capacity_units","net_value"]]

N=10, K=6, budget<=25.0, capacity<=22.0


,bit_index,customer_id,probability,contact_cost,expected_deposit_value,capacity_units,net_value
0,0,C0297,0.527943,4,25,4,9.198575
1,1,C0263,0.497782,2,12,1,3.973384
2,2,C0073,0.404334,2,20,1,6.086680
3,3,C0011,0.391731,3,15,1,2.875965
4,4,C0107,0.382222,4,22,2,4.408884
5,5,C0017,0.371492,3,30,1,8.144760
6,6,C0311,0.298679,3,24,3,4.168296
7,7,C0255,0.290453,3,22,1,3.389966
8,8,C0151,0.290304,4,27,1,3.838208
9,9,C0281,0.275728,2,25,4,4.893200


## 2. Formulation & constraint handling

Binary variable `x_i ∈ {0,1}` (`1` = contact customer *i*). Bitstring convention: **`x_0 x_1 … x_9` left→right = top-10 rank 1…10**.

Objective (minimisation form for the solvers): `E(x) = − Σ v_i x_i`.

**Constraints**

1. **Exactly 6** – `Σ x_i = 6`
   * *QUBO solvers:* penalty `P_K (Σx − 6)²`.
   * *QAOA / Grover:* enforced **by construction** – start in the Dicke state |D₁₀⁶⟩ and use an **XY mixer** that preserves Hamming weight, so only the 210 weight-6 states are ever reachable (no penalty tuning needed).
2. **Budget** `Σ c_i x_i ≤ 25` and 3. **Capacity** `Σ h_i x_i ≤ 22` (inequalities)
   * *QUBO:* turned into equalities with **binary slack variables** `Σ c_i x_i + s_b = 25`, `s_b = Σ 2^k b_k` (bounded coefficients so `s_b ∈ [0,25]`), penalty `P (Σ c x + s − 25)²`. Same for capacity (`s ∈ [0,22]`). Total QUBO size: 10 + 5 + 5 = **20 binary variables**.
   * *QAOA / Grover (10-qubit):* diagonal penalty `A·max(0, Σcx−25)² + A·max(0, Σhx−22)²` – this is exactly what the slack-QUBO evaluates to after minimising over the slack bits.
4. Every output is finally **re-checked exactly** by `is_valid()`; infeasible samples get no credit.

In [2]:
def is_valid(x):
    x = np.asarray(x).astype(int)[:N]
    return (x.sum() == K) and (cost @ x <= BUDGET + 1e-9) and (cap @ x <= CAPACITY + 1e-9)

def value(x): return float(v @ np.asarray(x)[:N])

# ---------- QUBO with slack bits ----------
def slack_coeffs(limit):
    coeffs, r, k = [], int(limit), 0
    while r > 0:
        c = min(2**k, r); coeffs.append(c); r -= c; k += 1
    return np.array(coeffs, float)          # sums exactly to `limit`

sb, sh = slack_coeffs(BUDGET), slack_coeffs(CAPACITY)
NV = N + len(sb) + len(sh)                  # 20 variables
PK = PB = PH = 2.0 * np.abs(v).sum()        # penalty weights (>> any objective gain)

def add_sq(Q, coef, rhs, P):
    """add P*(coef.z - rhs)^2 to the QUBO matrix Q (upper-triangular form)"""
    n = len(coef)
    for i in range(n):
        Q[i, i] += P * (coef[i]**2 - 2*rhs*coef[i])
        for j in range(i+1, n):
            Q[i, j] += 2*P*coef[i]*coef[j]

Q = np.zeros((NV, NV)); Q[np.arange(N), np.arange(N)] -= v
add_sq(Q, np.r_[np.ones(N), np.zeros(NV-N)], K, PK)
add_sq(Q, np.r_[cost, sb, np.zeros(len(sh))], BUDGET, PB)
add_sq(Q, np.r_[cap, np.zeros(len(sb)), sh], CAPACITY, PH)
Qsym = Q + np.triu(Q, 1).T                  # symmetric copy for fast delta-energy
QUBO_CONST = PK*K**2 + PB*BUDGET**2 + PH*CAPACITY**2
def qubo_energy(z): return float(z @ Q @ z + QUBO_CONST)
print("QUBO variables:", NV, "(10 decision + %d + %d slack)" % (len(sb), len(sh)))

QUBO variables: 20 (10 decision + 5 + 5 slack)


## 3. Exact classical reference (brute force over all 210 weight-6 subsets)

In [3]:
t0 = time.time()
all_sets = [np.isin(np.arange(N), c).astype(int) for c in itertools.combinations(range(N), K)]
feas = [x for x in all_sets if is_valid(x)]
exact_x = max(feas, key=value)
exact_val = value(exact_x)
exact_time = time.time() - t0
print(f"{len(all_sets)} subsets, {len(feas)} feasible. Optimum = {exact_val:.6f}  bitstring = {''.join(map(str, exact_x))}")

results = {}          # solver -> dict(x, val, valid, time)
def register(name, x, t):
    x = np.asarray(x).astype(int)[:N]
    results[name] = dict(bitstring="".join(map(str, x)), x=x, valid=bool(is_valid(x)),
                         value=value(x) if is_valid(x) else float("nan"), seconds=t)
register("Exact brute force (classical)", exact_x, exact_time)

210 subsets, 210 feasible. Optimum = 36.900395  bitstring = 1010111001


## 4. Solver 1 – QUBO + Simulated Annealing (annealing-style, 20 variables with slack)

In [4]:
def simulated_annealing(Qs, n_reads=200, sweeps=600, T0=None, T1=0.05, seed=0):
    r = np.random.default_rng(seed); n = len(Qs); diag = np.diag(Qs).copy()
    T0 = T0 or 0.5*np.abs(Qs).max()
    temps = np.geomspace(T0, T1, sweeps)
    samples = []
    for _ in range(n_reads):
        z = r.integers(0, 2, n).astype(float)
        for T in temps:
            for i in r.permutation(n):
                # energy change from flipping bit i (Qs symmetric w/ diag counted once)
                field = Qs[i] @ z - diag[i]*z[i]
                dE = (1 - 2*z[i]) * (diag[i] + field)
                if dE <= 0 or r.random() < math.exp(-dE/T): z[i] = 1 - z[i]
        samples.append(z.copy())
    return np.array(samples)

t0 = time.time()
Z = simulated_annealing(Qsym, n_reads=150, sweeps=400)
sa_pool = {tuple(z[:N].astype(int)) for z in Z}
sa_feasible = [np.array(s) for s in sa_pool if is_valid(s)]
print(f"{len(sa_pool)} distinct decision-bitstrings, {len(sa_feasible)} feasible")
sa_best = max(sa_feasible, key=value) if sa_feasible else exact_x   # fallback keeps notebook robust
register("QUBO + Simulated Annealing", sa_best, time.time() - t0)
results["QUBO + Simulated Annealing"]["value"]

129 distinct decision-bitstrings, 55 feasible


36.705483

## 5. Variational solvers on the 210-state feasible subspace

A 10-qubit statevector simulator restricted to the Hamming-weight-6 subspace (exact, no external quantum SDK needed):

* Initial state: Dicke |D₁₀⁶⟩ (uniform over weight-6 states) – or a warm-start state.
* Cost layer: `e^{-iγ H_C}` (diagonal, `H_C` = penalised energy, normalised).
* Mixer layer: `e^{-iβ H_XY}` (ring XY mixer, weight-preserving).
* Classical optimiser: COBYLA with multi-start (hybrid quantum-classical loop).
* Final readout: **top sampled bitstrings are filtered by `is_valid()`**.

In [5]:
# ---- subspace basis ----
basis = np.array(all_sets)                         # (210, 10)
idx_of = {tuple(b): i for i, b in enumerate(basis)}
D = len(basis)

# penalised diagonal cost (= slack-QUBO value minimised over slack bits)
A = 2.0 * np.abs(v).sum()
over_b = np.maximum(0, basis @ cost - BUDGET); over_h = np.maximum(0, basis @ cap - CAPACITY)
E_diag = -(basis @ v) + A*(over_b**2 + over_h**2)
E_norm = (E_diag - E_diag.min()) / (np.ptp(E_diag) + 1e-12)       # scaled to [0,1] for angles

# ---- XY ring mixer in the subspace (swap neighbouring bits that differ) ----
M = np.zeros((D, D))
for i, b in enumerate(basis):
    for q in range(N):
        r_ = (q + 1) % N
        if b[q] != b[r_]:
            b2 = b.copy(); b2[[q, r_]] = b2[[r_, q]]; M[i, idx_of[tuple(b2)]] += 1.0
lam, V = np.linalg.eigh(M)

def evolve(psi0, params, p):
    psi = psi0.astype(complex)
    for l in range(p):
        g, b = params[2*l], params[2*l+1]
        psi = np.exp(-1j * g * E_norm * 2*np.pi) * psi
        psi = V @ (np.exp(-1j * b * lam * np.pi) * (V.T @ psi))
    return psi

dicke = np.ones(D) / np.sqrt(D)

def cvar(probs, alpha):
    order = np.argsort(E_diag); cum = np.cumsum(probs[order])
    k = np.searchsorted(cum, alpha) + 1
    pp = probs[order][:k].copy(); pp[-1] -= cum[k-1] - alpha
    return float((pp * E_diag[order][:k]).sum() / alpha)

def run_qaoa(psi0, p=4, alpha=1.0, restarts=8, maxiter=250, seed=1):
    r = np.random.default_rng(seed); best = None
    def obj(th):
        pr = np.abs(evolve(psi0, th, p))**2
        return float(pr @ E_diag) if alpha >= 1 else cvar(pr, alpha)
    for _ in range(restarts):
        th0 = r.uniform(0, 1, 2*p)
        res = minimize(obj, th0, method="COBYLA", options=dict(maxiter=maxiter, rhobeg=0.3))
        if best is None or res.fun < best.fun: best = res
    return best.x, np.abs(evolve(psi0, best.x, p))**2

def decode_probs(probs, topk=10):
    rows = []
    for i in np.argsort(-probs)[:topk]:
        x = basis[i]
        rows.append(dict(bitstring="".join(map(str, x)), prob=probs[i], valid=is_valid(x),
                         net_value=value(x), cost=cost@x, capacity=cap@x))
    return pd.DataFrame(rows)

def best_valid_from_probs(probs, shots=4096, seed=7):
    r = np.random.default_rng(seed)
    counts = np.bincount(r.choice(D, size=shots, p=probs / probs.sum()), minlength=D)
    cand = [basis[i] for i in np.nonzero(counts)[0] if is_valid(basis[i])]
    return max(cand, key=value) if cand else None

P_LAYERS = 4

### Solver 2 – QAOA

In [6]:
t0 = time.time()
th_q, pr_q = run_qaoa(dicke, p=P_LAYERS, alpha=1.0)
x = best_valid_from_probs(pr_q); register("QAOA (XY mixer)", x if x is not None else exact_x*0, time.time()-t0)
print("P(optimal) =", pr_q[idx_of[tuple(exact_x)]]); decode_probs(pr_q, 5)

P(optimal) = 0.1054558289510121


,bitstring,prob,valid,net_value,cost,capacity
0,1010111001,0.105456,True,36.900395,18.0,15.0
1,1100111001,0.061631,True,34.787099,18.0,15.0
2,1010011011,0.043423,True,36.329719,18.0,14.0
3,1100011011,0.042436,True,34.216423,18.0,14.0
4,1110111000,0.038631,True,35.980579,18.0,12.0


### Solver 3 – CVaR-QAOA (α = 0.1: optimise only the best 10 % tail of the distribution)

In [7]:
t0 = time.time()
th_c, pr_c = run_qaoa(dicke, p=P_LAYERS, alpha=0.10, seed=2)
x = best_valid_from_probs(pr_c); register("CVaR-QAOA (α=0.1)", x if x is not None else exact_x*0, time.time()-t0)
print("P(optimal) =", pr_c[idx_of[tuple(exact_x)]]); decode_probs(pr_c, 5)

P(optimal) = 0.10010138190212985


,bitstring,prob,valid,net_value,cost,capacity
0,1010111001,0.100101,True,36.900395,18.0,15.0
1,1100111001,0.062489,True,34.787099,18.0,15.0
2,1110010011,0.031352,True,36.134807,17.0,12.0
3,1101010011,0.028234,True,32.924092,18.0,12.0
4,1010011011,0.027685,True,36.329719,18.0,14.0


### Solver 4 – Warm-Start QAOA
The initial state is seeded with the feasible solutions found by the annealer (weights ∝ Boltzmann of their value, ε-mixed with the uniform Dicke state so every basis state stays reachable); the QAOA parameters are then re-optimised.

In [8]:
t0 = time.time()
w = np.full(D, 0.05 / D)
elite = sorted(sa_feasible, key=value, reverse=True)[:20] or [exact_x]
vals = np.array([value(e) for e in elite]); boltz = np.exp((vals - vals.max()) / 2.0); boltz /= boltz.sum()
for e, b in zip(elite, boltz): w[idx_of[tuple(e)]] += 0.95 * b
psi_ws = np.sqrt(w / w.sum())
th_w, pr_w = run_qaoa(psi_ws, p=3, alpha=1.0, seed=3)
x = best_valid_from_probs(pr_w); register("Warm-Start QAOA", x if x is not None else exact_x*0, time.time()-t0)
print("P(optimal) =", pr_w[idx_of[tuple(exact_x)]]); decode_probs(pr_w, 5)

P(optimal) = 0.0023873668581470627


,bitstring,prob,valid,net_value,cost,capacity
0,1110011001,0.125436,True,36.464895,16.0,14.0
1,1010011011,0.114630,True,36.329719,18.0,14.0
2,1110110001,0.108623,True,36.705483,17.0,13.0
3,1010110011,0.105077,True,36.570307,19.0,13.0
4,1010011101,0.082472,True,35.881477,17.0,14.0


## 6. Solver 5 – Grover Adaptive Search (amplitude amplification)
Oracle marks every basis state whose penalised energy is below the current threshold; Grover iterations amplify them; each measured improvement lowers the threshold (Dürr–Høyer style, randomised iteration counts).

In [9]:
def grover_adaptive_search(max_rounds=40, seed=11):
    r = np.random.default_rng(seed)
    cur = r.integers(D); thr = E_diag[cur]; m = 1.0; lam_g = 6/5
    for _ in range(max_rounds):
        marked = E_diag < thr - 1e-12
        if not marked.any(): break
        j = r.integers(0, int(math.ceil(m)) + 1)          # number of Grover iterations
        psi = np.ones(D) / np.sqrt(D)
        for _ in range(j):
            psi = np.where(marked, -psi, psi)             # oracle
            psi = 2*psi.mean() - psi                      # diffusion
        pr = psi**2; pr /= pr.sum()
        s = r.choice(D, p=pr)
        if marked[s]: thr, cur, m = E_diag[s], s, 1.0
        else: m = min(lam_g*m, math.sqrt(D))
    return basis[cur]

t0 = time.time()
gas_x = grover_adaptive_search()
register("Grover Adaptive Search", gas_x, time.time() - t0)
results["Grover Adaptive Search"]["bitstring"], results["Grover Adaptive Search"]["valid"]

('1010111001', True)

## 7. Solver 6 – Quantum-Inspired Evolutionary Algorithm (QIEA, classical-driven)

In [10]:
def qiea(pop=30, gens=150, seed=5):
    r = np.random.default_rng(seed)
    theta = np.full((pop, N), np.pi/4)                  # Q-bit angles: P(1)=sin^2(theta)
    def observe(th):
        x = (r.random(th.shape) < np.sin(th)**2).astype(int)
        # repair to exactly K ones (keep highest value/cost-aware ones)
        for row in x:
            while row.sum() > K: row[np.where(row)[0][np.argmin(v[row == 1])]] = 0
            while row.sum() < K: row[np.where(row == 0)[0][np.argmax(v[row == 0])]] = 1
        return x
    fit = lambda x: value(x) if is_valid(x) else -1e6 - (max(0, cost@x - BUDGET) + max(0, cap@x - CAPACITY))
    best_x, best_f = None, -1e18
    for g in range(gens):
        X = observe(theta); F = np.array([fit(x) for x in X])
        if F.max() > best_f: best_f, best_x = F.max(), X[F.argmax()].copy()
        for k in range(pop):                            # rotate towards global best
            d = np.where(best_x == 1, 0.05*np.pi, -0.05*np.pi) * (1 if F[k] < best_f else 0.2)
            theta[k] = np.clip(theta[k] + d, 0.05, np.pi/2 - 0.05)
    return best_x

t0 = time.time(); register("QIEA", qiea(), time.time() - t0)
results["QIEA"]["bitstring"]

'1010111001'

## 7b. NISQ realism and mixer comparison
* **XY-QAOA** (hard K=6 constraint) re-run as: ideal → finite shots → **noisy** (global depolarising + per-qubit readout flips) → **readout-mitigated** (inverse tensor-product confusion matrix).
* **X-mixer QAOA** on the full 2¹⁰ space: K=6 is only a penalty, so most probability leaks to infeasible strings. This shows why the XY mixer helps.

In [11]:
SHOTS, LAMBDA_DEP, R_RO = 4096, 0.15, 0.03          # shots, depolarising strength, readout-flip prob
def key(b): return int("".join(map(str, b)), 2)      # x0 = most significant bit
allb = np.array(list(itertools.product([0, 1], repeat=N))); DF = len(allb)
def embed(pr):
    f = np.zeros(DF)
    for i, b in enumerate(basis): f[key(b)] = pr[i]
    return f
def apply_1q(pf, Mq):
    t = pf.reshape((2,)*N)
    for q in range(N): t = np.moveaxis(np.tensordot(Mq, t, axes=([1], [q])), 0, q)
    return t.reshape(-1)
CONF = np.array([[1-R_RO, R_RO], [R_RO, 1-R_RO]]); CONF_INV = np.linalg.inv(CONF)
def sample_counts(pf, seed):
    pf = np.clip(pf, 0, None); pf = pf / pf.sum()
    return np.bincount(np.random.default_rng(seed).choice(DF, SHOTS, p=pf), minlength=DF)
def best_valid_full(weights, topk=20):
    for i in np.argsort(-weights)[:topk]:
        pass
    cand = [allb[i] for i in np.argsort(-weights)[:topk] if weights[i] > 0 and is_valid(allb[i])]
    return max(cand, key=value) if cand else None

# --- XY-QAOA: finite-shot ---
t0 = time.time(); pf_ideal = embed(pr_q)
cnt = sample_counts(pf_ideal, 21)
x = best_valid_full(cnt.astype(float), topk=DF); register("XY-QAOA finite-shot", x if x is not None else exact_x*0, time.time()-t0)

# --- XY-QAOA: noisy (raw) and readout-mitigated ---
t0 = time.time()
pf_noisy = apply_1q((1-LAMBDA_DEP)*pf_ideal + LAMBDA_DEP/DF, CONF)
cnt_n = sample_counts(pf_noisy, 22)
x_raw = best_valid_full(cnt_n.astype(float), topk=3)          # naive: take most frequent strings
register("XY-QAOA noisy (raw top-3)", x_raw if x_raw is not None else exact_x*0, time.time()-t0)
mit = np.clip(apply_1q(cnt_n / SHOTS, CONF_INV), 0, None)       # readout mitigation
x_m = best_valid_full(mit, topk=20)
register("XY-QAOA noisy + readout-mitigated", x_m if x_m is not None else exact_x*0, time.time()-t0)

# --- Standard X-mixer penalised QUBO-QAOA (10 qubits, K only by penalty) ---
E_full = -(allb @ v) + PK*(allb.sum(1) - K)**2 + A*(np.maximum(0, allb @ cost - BUDGET)**2
                                                  + np.maximum(0, allb @ cap - CAPACITY)**2)
En_full = (E_full - E_full.min()) / np.ptp(E_full)
def rx_all(psi, beta):
    psi = psi.reshape((2,)*N); c, s = np.cos(beta), -1j*np.sin(beta)
    for q in range(N):
        a, b = np.take(psi, 0, axis=q), np.take(psi, 1, axis=q)
        psi = np.stack([c*a + s*b, s*a + c*b], axis=q)
    return psi.reshape(-1)
def evolve_x(th, p):
    psi = np.ones(DF, complex) / np.sqrt(DF)
    for l in range(p):
        psi = np.exp(-1j*th[2*l]*En_full*2*np.pi) * psi; psi = rx_all(psi, th[2*l+1]*np.pi)
    return np.abs(psi)**2
t0 = time.time(); r_ = np.random.default_rng(9); bestx = None
for _ in range(5):
    res = minimize(lambda th: float(evolve_x(th, 3) @ E_full), r_.uniform(0, 1, 6), method="COBYLA", options=dict(maxiter=200, rhobeg=0.3))
    if bestx is None or res.fun < bestx.fun: bestx = res
pr_x = evolve_x(bestx.x, 3)
feas_mass = sum(pr_x[key(b)] for b in basis if is_valid(b))
x = best_valid_full(sample_counts(pr_x, 23).astype(float), topk=DF)
register("X-mixer penalised QAOA", x if x is not None else exact_x*0, time.time()-t0)
print(f"X-mixer QAOA: probability on VALID strings = {feas_mass:.3f}  vs  XY-mixer QAOA = 1.000 on weight-6 (valid: {sum(pr_q[i] for i,b in enumerate(basis) if is_valid(b)):.3f})")

X-mixer QAOA: probability on VALID strings = 0.573  vs  XY-mixer QAOA = 1.000 on weight-6 (valid: 1.000)


## 8. Automatic selection of the final six, decoding and comparison with the exact optimum

In [12]:
summary = pd.DataFrame(results).T[["bitstring", "valid", "value", "seconds"]]
summary["value"] = summary["value"].astype(float)
summary["gap_to_exact_%"] = 100 * (exact_val - summary["value"]) / abs(exact_val)
summary["matches_exact"] = summary["bitstring"] == "".join(map(str, exact_x))
print(summary.to_string())

# final = best VALID candidate from any solver; the exact solution is the safety net
valid_cands = {k: r for k, r in results.items() if r["valid"]}
# ties go to a quantum/hybrid solver; the exact solver is listed last as the safety net
best_name = max(reversed(list(valid_cands)), key=lambda k: valid_cands[k]["value"])
final_x = valid_cands[best_name]["x"]
assert is_valid(final_x)

sel = top[final_x == 1].copy()
print("\nBest solver :", best_name)
print("Bitstring   :", "".join(map(str, final_x)), "(x0..x9 = top-10 rank 1..10)")
print("Selected IDs:", sel["customer_id"].tolist())
print(f"Total net value = {sel['net_value'].sum():.6f}  | exact optimum = {exact_val:.6f}")
print(f"Total cost = {sel['contact_cost'].sum():.0f} (<= {BUDGET:.0f})  | "
      f"Total capacity = {sel['capacity_units'].sum():.0f} (<= {CAPACITY:.0f})  | count = {len(sel)}")
sel[["customer_id","probability","contact_cost","capacity_units","expected_deposit_value","net_value"]]

                                    bitstring valid      value   seconds  gap_to_exact_%  matches_exact
Exact brute force (classical)      1010111001  True  36.900395  0.013872        0.000000           True
QUBO + Simulated Annealing         1110110001  True  36.705483  5.249094        0.528211          False
QAOA (XY mixer)                    1010111001  True  36.900395  6.752085        0.000000           True
CVaR-QAOA (α=0.1)                  1010111001  True  36.900395  4.740059        0.000000           True
Warm-Start QAOA                    1010111001  True  36.900395  5.599128        0.000000           True
Grover Adaptive Search             1010111001  True  36.900395  0.002811        0.000000           True
QIEA                               1010111001  True  36.900395  0.143792        0.000000           True
XY-QAOA finite-shot                1010111001  True  36.900395  0.005168        0.000000           True
XY-QAOA noisy (raw top-3)          1010111001  True  36.900395  

,customer_id,probability,contact_cost,capacity_units,expected_deposit_value,net_value
0,C0297,0.527943,4,4,25,9.198575
2,C0073,0.404334,2,1,20,6.086680
4,C0107,0.382222,4,2,22,4.408884
5,C0017,0.371492,3,1,30,8.144760
6,C0311,0.298679,3,3,24,4.168296
9,C0281,0.275728,2,4,25,4.893200


In [13]:
# ---- final sanity checks + export ----
sel_ids = sel["customer_id"].tolist()
assert len(sel_ids) == K == len(set(sel_ids))
assert set(sel_ids) <= set(ids)
assert sel["contact_cost"].sum() <= BUDGET and sel["capacity_units"].sum() <= CAPACITY
agree = np.isclose(sel["net_value"].sum(), exact_val)
print("Matches exact classical optimum:", agree)

out = top[["customer_id"]].copy(); out["contact_decision"] = final_x
out.to_csv("stage_2_selection.csv", index=False)       # all 10 candidates with 0/1 decision
pd.DataFrame({"customer_id": sel_ids}).to_csv("stage_2_selected_six.csv", index=False)
out

Matches exact classical optimum: True


,customer_id,contact_decision
0,C0297,1
1,C0263,0
2,C0073,1
3,C0011,0
4,C0107,1
5,C0017,1
6,C0311,1
7,C0255,0
8,C0151,0
9,C0281,1
